In [1]:
import pandas as pd
import numpy as np

def final_dataset_cleaning(input_csv: str, output_csv: str):
    print("Membaca dataset masif...")
    input_csv = (r"C:\Users\HP Victus\Documents\MARKET PREDICT PROJECT\DATA_INGESTION\btc_1m_pure_features_massive.csv")
    df = pd.read_csv(input_csv)
    
    # 1. Pastikan timestamp menjadi index, bukan kolom fitur (karena ML tidak bisa baca string waktu)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # 2. Bikin ulang kolom target (jika sebelumnya terlewat tidak tersimpan)
    # Target: Apakah harga Close di menit ke-depan (shift -1) lebih tinggi dari Close saat ini?
    df["target_return"] = np.log(df["close"].shift(-1) / df["close"])
    df["target_signal"] = (df["target_return"] > 0).astype(int)
    
    # 3. Hapus 240 baris pertama (Periode Warm-Up untuk VWAP 4H dan Rolling 60m)
    # Ini otomatis melenyapkan semua nilai 0.0 palsu dan nilai kosong (,,) di awal data
    df_clean = df.iloc[240:].copy()
    
    # 4. Hapus baris paling akhir karena target_return-nya pasti NaN (tidak ada masa depan untuk dihitung)
    df_clean = df_clean.dropna(subset=['target_return'])
    
    df_clean.to_csv(output_csv)
    print(f"Cleaning Selesai! Data siap masuk XGBoost. Total Baris Bersih: {len(df_clean)}")

if __name__ == "__main__":
    final_dataset_cleaning("btc_1m_pure_features_massive.csv", "btc_1m_ml_ready.csv")

Membaca dataset masif...
Cleaning Selesai! Data siap masuk XGBoost. Total Baris Bersih: 3022319


EDA

In [2]:
import pandas as pd
import numpy as np

def run_ml_diagnostic(csv_path: str):
    print(f"Memuat dataset: {csv_path}...")
    df = pd.read_csv(csv_path, index_col='timestamp', parse_dates=True)
    
    print("\n" + "="*50)
    print("🔬 HASIL DIAGNOSTIK DATASET ML")
    print("="*50)
    
    # 1. Cek Ukuran Data
    print(f"Total Baris : {len(df):,}")
    print(f"Total Kolom : {df.shape[1]}")
    
    # 2. Cek Keseimbangan Kelas Target
    target_dist = df['target_signal'].value_counts(normalize=True) * 100
    print(f"\n⚖️ Keseimbangan Target:")
    print(f"  Label 0 (Turun) : {target_dist.get(0, 0):.2f}%")
    print(f"  Label 1 (Naik)  : {target_dist.get(1, 0):.2f}%")
    if abs(target_dist.get(0, 0) - target_dist.get(1, 0)) > 10:
        print("  ⚠️ PERINGATAN: Target Imbalance terdeteksi!")
    else:
        print("  ✅ Target seimbang (Normal untuk timeframe 1m).")
        
    # 3. Cek NaN dan Infinity
    # Ubah inf menjadi NaN untuk memudahkan perhitungan
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_counts = df.isna().sum()
    total_nans = nan_counts.sum()
    
    print(f"\n🕳️ Pengecekan NaN & Infinity:")
    if total_nans == 0:
        print("  ✅ Dataset 100% bersih dari NaN dan Infinity.")
    else:
        print(f"  ⚠️ Ditemukan {total_nans:,} nilai kosong/infinity pada kolom:")
        print(nan_counts[nan_counts > 0])
        
    # 4. Ringkasan Statistik untuk Deteksi Outlier (Pilih kolom krusial)
    print("\n📊 Statistik Deteksi Outlier (Min vs Max):")
    sample_cols = ['volume_delta', 'vwap_zscore_4h', 'oi_change_1m', 'target_return']
    for col in sample_cols:
        if col in df.columns:
            print(f"  {col}: Min {df[col].min():.4f} | Max {df[col].max():.4f}")

if __name__ == "__main__":
    run_ml_diagnostic("btc_1m_ml_ready.csv")

Memuat dataset: btc_1m_ml_ready.csv...

🔬 HASIL DIAGNOSTIK DATASET ML
Total Baris : 3,022,319
Total Kolom : 31

⚖️ Keseimbangan Target:
  Label 0 (Turun) : 50.84%
  Label 1 (Naik)  : 49.16%
  ✅ Target seimbang (Normal untuk timeframe 1m).

🕳️ Pengecekan NaN & Infinity:
  ⚠️ Ditemukan 11,586 nilai kosong/infinity pada kolom:
premium_index_bps    11586
dtype: int64

📊 Statistik Deteksi Outlier (Min vs Max):
  volume_delta: Min -16393.5160 | Max 12286.6540
  vwap_zscore_4h: Min -9.9648 | Max 9.6545
  oi_change_1m: Min -333.0266 | Max 333.0863
  target_return: Min -0.0651 | Max 0.0574


In [3]:
import pandas as pd

def patch_missing_values(csv_path: str):
    print("Menambal nilai kosong...")
    df = pd.read_csv(csv_path, index_col='timestamp', parse_dates=True)
    
    # Isi kekosongan dengan nilai menit sebelumnya (Forward Fill)
    # Jika masih ada NaN di baris paling awal, isi dengan 0.0 (netral)
    df['premium_index_bps'] = df['premium_index_bps'].ffill().fillna(0.0)
    
    df.to_csv(csv_path)
    print("Patching selesai. Dataset sekarang 100% sempurna!")

if __name__ == "__main__":
    patch_missing_values("btc_1m_ml_ready.csv")

Menambal nilai kosong...
Patching selesai. Dataset sekarang 100% sempurna!
